In [1]:
!pip install -q transformers datasets accelerate scikit-learn xgboost pandas numpy

import os
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    AutoModelForSequenceClassification
)
from datasets import Dataset
import xgboost as xgb
import torch

# Mount Google Drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/My Drive/NLPLabs-2024/Dont_Patronize_Me_Trainingset/'
DATA_FILE = os.path.join(BASE_PATH, "dontpatronizeme_pcl.tsv")

# Load data
df = pd.read_csv(DATA_FILE, sep='\t', skiprows=4, names=['id', 'art_id', 'keyword', 'country', 'text', 'label'])
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(int)

# Binarize labels
df['binary_label'] = df['label'].apply(lambda x: 1 if x > 1 else 0)

print(f"Total rows loaded: {len(df)}")
print(df['binary_label'].value_counts())

Mounted at /content/drive
Total rows loaded: 10469
binary_label
0    9476
1     993
Name: count, dtype: int64


In [3]:

MODEL_NAME = "roberta-base"
TAPT_OUTPUT_DIR = os.path.join(BASE_PATH, "tapt_roberta_base")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

text_dataset = Dataset.from_pandas(df[['text']])

def tokenize_function(examples):
    return tokenizer(examples["text"], return_special_tokens_mask=True, truncation=True, max_length=128)

tokenized_datasets = text_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# The Data Collator randomly masks 15% of the words
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

tapt_args = TrainingArguments(
    output_dir=TAPT_OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    save_steps=500,
    save_total_limit=1,
    prediction_loss_only=True,
    learning_rate=2e-5,
    fp16=True,
)

tapt_trainer = Trainer(
    model=model_mlm,
    args=tapt_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets,
)

print("Starting Task-Adaptive Pretraining (TAPT)...")
tapt_trainer.train()

tapt_trainer.save_model(TAPT_OUTPUT_DIR)
tokenizer.save_pretrained(TAPT_OUTPUT_DIR)
print(f"TAPT Model saved to {TAPT_OUTPUT_DIR}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Map:   0%|          | 0/10469 [00:00<?, ? examples/s]

Starting Task-Adaptive Pretraining (TAPT)...


Step,Training Loss
500,1.721827
1000,1.626948
1500,1.610112


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TAPT Model saved to /content/drive/My Drive/NLPLabs-2024/Dont_Patronize_Me_Trainingset/tapt_roberta_base


In [5]:
!pip install -q textaugment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 99.6 MB/s eta 0:00:00


In [10]:
import pandas as pd
import numpy as np
import torch
import os
from tqdm import tqdm
from textaugment import EDA
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import MarianMTModel, MarianTokenizer
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# Load Augmentation Models
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading MarianMT Translation Models...")
en_fr_name = "Helsinki-NLP/opus-mt-en-fr"
fr_en_name = "Helsinki-NLP/opus-mt-fr-en"

en_fr_tokenizer = MarianTokenizer.from_pretrained(en_fr_name)
en_fr_model = MarianMTModel.from_pretrained(en_fr_name).to(device)
fr_en_tokenizer = MarianTokenizer.from_pretrained(fr_en_name)
fr_en_model = MarianMTModel.from_pretrained(fr_en_name).to(device)

t_eda = EDA()

def back_translate_batch(texts):
    inputs = en_fr_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    translated = en_fr_model.generate(**inputs)
    fr_texts = [en_fr_tokenizer.decode(t, skip_special_tokens=True) for t in translated]

    inputs_back = fr_en_tokenizer(fr_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    back_translated = fr_en_model.generate(**inputs_back)
    return [fr_en_tokenizer.decode(t, skip_special_tokens=True) for t in back_translated]


Loading MarianMT Translation Models...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
from tqdm import tqdm

# Filter all PCL samples from the original dataframe
pcl_samples = df[df['binary_label'] == 1].copy()
original_texts = pcl_samples['text'].tolist()

print(f"Pre-computing back-translations for {len(original_texts)} samples...")
bt_results = []
batch_size = 16

for i in tqdm(range(0, len(original_texts), batch_size)):
    batch = original_texts[i:i+batch_size]
    bt_results.extend(back_translate_batch(batch))

# Create a clean dataframe of augmented samples
aug_df = pd.DataFrame({
    'original_id': pcl_samples['id'].values,
    'text': bt_results,
    'binary_label': [1] * len(bt_results)
})

# Save to Drive
aug_df.to_csv(os.path.join(BASE_PATH, "precomputed_backtranslations.csv"), index=False)
print("Saved all back-translations to Drive!")

In [11]:

#Setup Cross-Validation & Trackers
CUSTOM_MODEL_PATH = TAPT_OUTPUT_DIR
BEST_MODEL_SAVE_DIR = os.path.join(BASE_PATH, "Best_TAPT_Classifier")

oof_probs = np.zeros(len(df))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


best_fold_f1 = 0.0
best_fold_idx = -1

for fold, (train_idx, val_idx) in enumerate(skf.split(df['text'], df['binary_label'])):
    print(f"\n{'='*40}\n--- Starting Fold {fold + 1}/5 ---\n{'='*40}")

    # prevents data leakage
    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()

    # augment training fold
    pcl_df = train_df[train_df['binary_label'] == 1].copy()
    texts_to_aug = pcl_df['text'].tolist()

    print(f"Augmenting {len(texts_to_aug)} PCL samples via Back-Translation...")
    bt_texts = []
    batch_size = 16
    for i in tqdm(range(0, len(texts_to_aug), batch_size)):
        batch = texts_to_aug[i:i+batch_size]
        bt_texts.extend(back_translate_batch(batch))

    print("Performing EDA (Synonym Replacement)...")
    eda_texts = [t_eda.synonym_replacement(text) for text in texts_to_aug]

    df_aug_bt = pd.DataFrame({'text': bt_texts, 'binary_label': [1]*len(bt_texts)})
    df_aug_eda = pd.DataFrame({'text': eda_texts, 'binary_label': [1]*len(eda_texts)})
    train_df = pd.concat([train_df, df_aug_bt, df_aug_eda], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

    # tokenise
    train_ds = Dataset.from_pandas(train_df[['text', 'binary_label']].rename(columns={'binary_label': 'label'}))
    val_ds = Dataset.from_pandas(val_df[['text', 'binary_label']].rename(columns={'binary_label': 'label'}))

    def tokenize_clf(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

    train_ds = train_ds.map(tokenize_clf, batched=True)
    val_ds = val_ds.map(tokenize_clf, batched=True)


    model_clf = AutoModelForSequenceClassification.from_pretrained(CUSTOM_MODEL_PATH, num_labels=2)

    for name, param in model_clf.named_parameters():
        if "encoder.layer.8" in name or "encoder.layer.9" in name or \
           "encoder.layer.10" in name or "encoder.layer.11" in name or "classifier" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        learning_rate=2e-5,
        weight_decay=0.01,
        fp16=True,
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model_clf,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )

    trainer.train()

    # track best model
    preds = trainer.predict(val_ds)
    probs = torch.nn.functional.softmax(torch.tensor(preds.predictions), dim=-1).numpy()[:, 1]
    oof_probs[val_idx] = probs

    fold_preds_binary = (probs >= 0.5).astype(int)
    fold_f1 = f1_score(val_df['binary_label'], fold_preds_binary)

    print(f"Fold {fold+1} Validation F1: {fold_f1:.4f}")

    if fold_f1 > best_fold_f1:
        print(f"New Best Model Found (F1 improved from {best_fold_f1:.4f} to {fold_f1:.4f}).")
        best_fold_f1 = fold_f1
        best_fold_idx = fold
        trainer.save_model(BEST_MODEL_SAVE_DIR)
        tokenizer.save_pretrained(BEST_MODEL_SAVE_DIR)

np.save(os.path.join(BASE_PATH, "roberta_tapt_aug_oof_probs.npy"), oof_probs)
print(f"\nOOF Generation Complete. The best classifier (Fold {best_fold_idx+1} with F1: {best_fold_f1:.4f}) ")


--- Starting Fold 1/5 ---
Augmenting 795 PCL samples via Back-Translation...


Performing EDA (Synonym Replacement)...


Map:   0%|          | 0/9965 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Map:   0%|          | 0/9966 [00:00<?, ? examples/s]

Map:   0%|          | 0/2093 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.351349,0.228950
2,0.231245,0.278664
3,0.182754,0.285403


Fold 5 Validation F1: 0.5168

OOF Generation Complete. The best classifier (Fold 4 with F1: 0.6039) 


In [15]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report

print("Building Meta-Dataset for XGBoost...")

roberta_probs = np.load(os.path.join(BASE_PATH, "roberta_tapt_aug_oof_probs.npy"))

meta_df = pd.DataFrame({
    'roberta_prob': roberta_probs,
    'keyword': df['keyword'],
    'label': df['binary_label']
})

meta_df['keyword_code'] = meta_df['keyword'].astype('category').cat.codes

X = meta_df[['roberta_prob', 'keyword_code']] # Add 'deberta_prob' here later
y = meta_df['label']

xgb_oof_preds = np.zeros(len(meta_df))
xgb_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in xgb_skf.split(X, y):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    clf = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3, # Keep shallow to prevent overfitting
        scale_pos_weight=5,
        eval_metric='logloss',
        random_state=42
    )

    clf.fit(X_tr, y_tr)
    xgb_oof_preds[val_idx] = clf.predict(X_va)


print("\n" + "="*40)
print("META-ENSEMBLE RESULTS")
print("="*40)

final_f1 = f1_score(y, xgb_oof_preds)
print(f"\nOVERALL F1-SCORE: {final_f1:.4f} 🚀\n")

print("--- CONFUSION MATRIX ---")
cm = confusion_matrix(y, xgb_oof_preds)
print(f"True Negatives (TN) : {cm[0][0]}")
print(f"False Positives (FP): {cm[0][1]}  <-- Let's see if we lowered this!")
print(f"False Negatives (FN): {cm[1][0]}")
print(f"True Positives (TP) : {cm[1][1]}\n")


print("--- KEYWORD PERFORMANCE ---")
meta_df['final_prediction'] = xgb_oof_preds
keyword_results = []

for kw in meta_df['keyword'].unique():
    kw_mask = meta_df['keyword'] == kw
    kw_y_true = meta_df[kw_mask]['label']
    kw_y_pred = meta_df[kw_mask]['final_prediction']

    kw_f1 = f1_score(kw_y_true, kw_y_pred, zero_division=0)
    keyword_results.append({'Keyword': kw, 'F1-Score': kw_f1})

kw_results_df = pd.DataFrame(keyword_results).sort_values(by='F1-Score', ascending=False)
print(kw_results_df.to_string(index=False))

Building Meta-Dataset for XGBoost...

META-ENSEMBLE RESULTS

OVERALL F1-SCORE: 0.5412 🚀

--- CONFUSION MATRIX ---
True Negatives (TN) : 8466
False Positives (FP): 1010  <-- Let's see if we lowered this!
False Negatives (FN): 250
True Positives (TP) : 743

--- KEYWORD PERFORMANCE ---
      Keyword  F1-Score
      in-need  0.619048
     homeless  0.612159
   vulnerable  0.572727
     hopeless  0.561605
poor-families  0.526316
      refugee  0.516949
        women  0.478873
    immigrant  0.426966
     disabled  0.414634
      migrant  0.336449


In [21]:
import os
import torch
import numpy as np
import pandas as pd
import xgboost as xgb
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import Dataset
from sklearn.metrics import f1_score, confusion_matrix, classification_report

print("--- STARTING FINAL SUBMISSION GENERATION & ANALYSIS ---")

# Setup Folders
SUBMISSION_DIR = os.path.join(BASE_PATH, "Grandmaster_Submissions")
os.makedirs(SUBMISSION_DIR, exist_ok=True)

#Train the FINAL XGBoost Judge
roberta_probs = np.load(os.path.join(BASE_PATH, "roberta_tapt_aug_oof_probs.npy"))
meta_train = pd.DataFrame({'roberta_prob': roberta_probs, 'keyword': df['keyword'], 'label': df['binary_label']})
inverse_keyword_mapping = {v: k for k, v in dict(enumerate(meta_train['keyword'].astype('category').cat.categories)).items()}
meta_train['keyword_code'] = meta_train['keyword'].map(inverse_keyword_mapping)

final_xgb = xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, scale_pos_weight=5, eval_metric='logloss', random_state=42)
final_xgb.fit(meta_train[['roberta_prob', 'keyword_code']], meta_train['label'])

BEST_MODEL_DIR = os.path.join(BASE_PATH, "Best_TAPT_Classifier")
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
model_clf = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_DIR).to("cuda" if torch.cuda.is_available() else "cpu")
model_clf.eval()

# Helper for Official Dev Set Filtering
def get_official_dev_data():
    full_df = pd.read_csv(os.path.join(BASE_PATH, "dontpatronizeme_pcl.tsv"), sep='\t', skiprows=4, names=['id', 'art_id', 'keyword', 'country', 'text', 'label'])
    full_df['binary_label'] = full_df['label'].apply(lambda x: 1 if x > 1 else 0)
    dev_ids = pd.read_csv(os.path.join(BASE_PATH, "dev_semeval_parids-labels.csv"))['par_id'].tolist()
    dev_df = full_df[full_df['id'].isin(dev_ids)].copy()
    dev_df['id'] = dev_df['id'].astype(int)
    dev_df = dev_df.set_index('id').loc[dev_ids].reset_index()
    return dev_df

def generate_outputs(df_to_predict, filename, has_labels=True):
    df_to_predict['text'] = df_to_predict['text'].astype(str)
    df_to_predict['keyword_code'] = df_to_predict['keyword'].map(inverse_keyword_mapping).fillna(-1)

    ds = Dataset.from_pandas(df_to_predict[['text']]).map(lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=128), batched=True)
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])
    dataloader = DataLoader(ds, batch_size=32)

    probs = []
    with torch.no_grad():
        for batch in dataloader:
            inputs = {k: v.to(model_clf.device) for k, v in batch.items()}
            outputs = model_clf(**inputs)
            probs.extend(torch.nn.functional.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy())

    final_preds = final_xgb.predict(pd.DataFrame({'roberta_prob': probs, 'keyword_code': df_to_predict['keyword_code']})[['roberta_prob', 'keyword_code']])

    with open(os.path.join(SUBMISSION_DIR, filename), 'w') as f:
        for p in final_preds: f.write(f"{int(p)}\n")

    return final_preds


# Process Dev Set
official_dev = get_official_dev_data()
dev_preds = generate_outputs(official_dev, "dev.txt", has_labels=True)

# Process Test Set
test_df = pd.read_csv(os.path.join(BASE_PATH, "task4_test.tsv"), sep='\t', header=None, names=['id', 'art_id', 'keyword', 'country', 'text'])
_ = generate_outputs(test_df, "test.txt", has_labels=False)


dev_true = official_dev['binary_label']
tn, fp, fn, tp = confusion_matrix(dev_true, dev_preds).ravel()
print(f"\n--- CONFUSION MATRIX ---")
print(f"True Negatives (TN) : {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP) : {tp}")

print("\n--- KEYWORD PERFORMANCE ---")
official_dev['final_prediction'] = dev_preds
kw_stats = []
for kw in official_dev['keyword'].unique():
    mask = official_dev['keyword'] == kw
    kw_f1 = f1_score(official_dev[mask]['binary_label'], official_dev[mask]['final_prediction'], zero_division=0)
    kw_stats.append({'Keyword': kw, 'F1-Score': kw_f1})

print(pd.DataFrame(kw_stats).sort_values(by='F1-Score', ascending=False).to_string(index=False))

print(f"\nSUBMISSION READY: Files are in {SUBMISSION_DIR}")

--- STARTING FINAL SUBMISSION GENERATION & ANALYSIS ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Map:   0%|          | 0/3832 [00:00<?, ? examples/s]


--- CONFUSION MATRIX ---
True Negatives (TN) : 1741
False Positives (FP): 154
False Negatives (FN): 14
True Positives (TP) : 185

--- KEYWORD PERFORMANCE ---
      Keyword  F1-Score
      in-need  0.750000
poor-families  0.729167
      refugee  0.727273
   vulnerable  0.692308
     homeless  0.683544
     disabled  0.666667
    immigrant  0.636364
      migrant  0.625000
     hopeless  0.623377
        women  0.615385

SUBMISSION READY: Files are in /content/drive/My Drive/NLPLabs-2024/Dont_Patronize_Me_Trainingset/Grandmaster_Submissions
